In [1]:
# ============================================================================
# PRE-PROCESSING PIPELINE FOR COSMOS2025 QUIESCENT GALAXY CLASSIFICATION
# ============================================================================
# This notebook handles the pre-processing workflow:
#   1. Data loading and initial quality assessment
#   2. Missing value imputation using MissForest
#   3. Noise injection into SAM photometry for domain adaptation
# ============================================================================

In [35]:
import pandas as pd
import numpy as np
import itertools
from sklearn.ensemble import RandomForestRegressor
from missforest import MissForest
import warnings
warnings.filterwarnings('ignore')

In [36]:
# ============================================================================
# CONFIGURATION
# ============================================================================
# Define the photometric bands used in the analysis
PHOTOMETRIC_BANDS = [
    'mag_model_hst-f814w',      # HST/ACS F814W
    'mag_model_uvista-y',       # VISTA/VIRCAM Y
    'mag_model_uvista-j',       # VISTA/VIRCAM J
    'mag_model_uvista-h',       # VISTA/VIRCAM H
    'mag_model_uvista-ks',      # VISTA/VIRCAM Ks
    'mag_model_irac-ch1',       # Spitzer/IRAC Channel 1
    'mag_model_irac-ch2',       # Spitzer/IRAC Channel 2
    'mag_model_f115w',          # JWST/NIRCam F115W
    'mag_model_f150w',          # JWST/NIRCam F150W
    'mag_model_f227w',          # JWST/NIRCam F277W
    'mag_model_f444w'           # JWST/NIRCam F444W
]

In [37]:
# ============================================================================
# 1. LOAD OBSERVATIONAL DATA
# ============================================================================
print("=" * 60)
print("STEP 1: Loading COSMOS2025 observational data")
print("=" * 60)

STEP 1: Loading COSMOS2025 observational data


In [38]:
# Load the pre-filtered COSMOS2025 sample
# (already selected by: F444W < 28, 2.5 < z < 5, log(M*/Msun) > 9.5)
cosmos_df = pd.read_csv('COSMOS2025_with_missing_values.csv')

In [39]:
# ============================================================================
# 2. Missing value imputation
# ============================================================================
print("\n" + "=" * 60)
print("STEP 2: Missing value imputation using MissForest")
print("=" * 60)


STEP 2: Missing value imputation using MissForest


In [40]:
# Extract magnitude columns only
cosmos_df_mag = cosmos_df.filter(regex='^mag_model_')

In [41]:
# Calculate and display missing fractions for each band
missing_fractions = cosmos_df_mag.isna().sum() / len(cosmos_magnitudes)
print("Missing value fractions per band:")
for band, frac in missing_fractions.items():
    band_short = band.replace('mag_model_', '')
    print(f"  {band_short:25s}: {frac*100:5.1f}%")

Missing value fractions per band:
  hst-f814w                :   4.1%
  uvista-y                 :   5.9%
  uvista-j                 :   3.4%
  uvista-h                 :   1.5%
  uvista-ks                :   0.9%
  irac-ch1                 :   1.5%
  irac-ch2                 :   1.4%
  f115w                    :   3.8%
  f150w                    :   1.3%
  f277w                    :   0.0%
  f444w                    :   0.0%


In [42]:
# Initialize MissForest with 50 iterations for convergence
# Uses default Random Forest hyperparameters (n_estimators=100, max_depth=None,
# min_samples_split=2, min_samples_leaf=1, random_state=0)
imputer = MissForest(max_iter=50)
# Perform imputation on the magnitude columns
print("Running MissForest imputation (this may take a few minutes)...")
cosmos_df_imputed_mag = imputer.fit_transform(cosmos_df_mag)
print("Imputation complete.")

Running MissForest imputation (this may take a few minutes)...


 26%|█████████████████████▎                                                            | 13/50 [00:13<00:39,  1.06s/it]

Imputation complete.


In [43]:
# Verify that all missing values have been filled
remaining_missing = cosmos_df_imputed_mag.isna().sum().sum()
print(f"Remaining missing values after imputation: {remaining_missing}")

Remaining missing values after imputation: 0


In [44]:
# Update the original dataframe with imputed magnitudes
cosmos_df.update(cosmos_df_imputed_mag)
print("Updated COSMOS dataframe with imputed magnitudes.")

Updated COSMOS dataframe with imputed magnitudes.


In [45]:
# ============================================================================
# 3. Noise Injection
# ============================================================================
print("\n" + "=" * 80)
print("STEP 3: Noise injection into SAM photometry for domain adaptation")
print("=" * 80)


STEP 3: Noise injection into SAM photometry for domain adaptation


In [46]:
# --- 3a. Compute multiplicative errors ---
# Multiplicative error = flux_err / flux, used as a quality indicator
flux_columns = [col for col in cosmos_df.columns if col.startswith('flux_model_')]

# Calculate multiplicative errors for each band
for flux_col in flux_columns:
    # Extract band name (e.g., 'cfht-u' from 'flux_model_cfht-u')
    band = flux_col.split('flux_model_')[1]
    
    # Corresponding error column
    err_col = f'flux_err-cal_model_{band}'
    
    # Calculate multiplicative error and create new column
    if err_col in cosmos_df.columns:
        cosmos_df[f'mult_err_{band}'] = cosmos_df[err_col] / cosmos_df[flux_col]
    else:
        print(f"Warning: Error column {err_col} not found for band {band}")

# Display the new columns created
new_columns = [col for col in cosmos_df.columns if col.startswith('mult_err_')]
print("New multiplicative error columns created:")
print(new_columns)

New multiplicative error columns created:
['mult_err_hst-f814w', 'mult_err_uvista-y', 'mult_err_uvista-j', 'mult_err_uvista-h', 'mult_err_uvista-ks', 'mult_err_irac-ch1', 'mult_err_irac-ch2', 'mult_err_f115w', 'mult_err_f150w', 'mult_err_f277w', 'mult_err_f444w']


In [47]:
# --- 3b. Apply multiplicative error threshold ---
cols = ['mult_err_hst-f814w', 'mult_err_uvista-y',
        'mult_err_uvista-j', 'mult_err_uvista-h', 'mult_err_uvista-ks',
        'mult_err_irac-ch1', 'mult_err_irac-ch2', 'mult_err_f115w',
        'mult_err_f150w', 'mult_err_f277w', 'mult_err_f444w']

mask = (cosmos_df[cols] <= 5).all(axis=1)

cosmos_df = cosmos_df[mask]

In [48]:
# --- 3c. Convert multiplicative errors to magnitude errors ---
# For small errors: mag_err ≈ 1.0857 * (flux_err / flux)
# The factor 1.0857 = 2.5 / ln(10)
print("Converting flux errors to magnitude errors...")
for item in cols:
    cosmos_df[item+'magerr'] = 1.0857 * cosmos_df[item]

Converting flux errors to magnitude errors...


In [49]:
# --- 3d. Compute color errors via error propagation ---
print("Computing color errors...")
bands = cosmos_df.columns[-22:-11]
for band1, band2 in itertools.combinations(bands, 2):
    color_name = f"{band1}-{band2}"
    cosmos_df[color_name+'_err'] = np.sqrt(cosmos_df[band1+'magerr']**2 + cosmos_df[band2+'magerr']**2)
cosmos_df_color_err = cosmos_df.iloc[:,-55:]

Computing color errors...


In [51]:
# --- 3e. Compute observed colors ---
print("Computing COSMOS observed colors...")
bands = cosmos_df.columns[2:13]
for band1, band2 in itertools.combinations(bands, 2):
    color_name = f"{band1}-{band2}"
    cosmos_df[color_name] = cosmos_df[band1] - cosmos_df[band2]
cosmos_df_color = cosmos_df.iloc[:,-55:]

Computing COSMOS observed colors...


In [52]:
# --- 3f. Compute SAM colors ---
print("Loading and processing SAM mock photometry...")
sam_df = pd.read_csv('SAM_Mags.csv')
bands = sam_df.columns
for band1, band2 in itertools.combinations(bands, 2):
    color_name = f"{band1}-{band2}"
    sam_df[color_name] = sam_df[band1] - sam_df[band2]
sam_df_color = sam_df.iloc[:, -55:]

Loading and processing SAM mock photometry...


In [58]:
# --- 3g. Injecting Noise by RandomForestRegressor ---
for i in range(55):
    colori = np.array(cosmos_df_color.iloc[:,i]).reshape(-1,1)
    color_erri = cosmos_df_color_err.iloc[:,i]
    sami = np.array(sam_df_color.iloc[:,i]).reshape(-1,1)
    
    rfi = RandomForestRegressor()
    rfi.fit(colori, color_erri)
    
    sed_erri = rfi.predict(sami)
    sam_df[f'{i}_err'] = sed_erri
    sigma = np.clip(np.abs(sed_erri), 1e-4, 1.0)

    delta_fi = np.random.normal(loc = 0, scale = sigma)
    
    sam_df[f'{sam_df_color.columns[i]}_noisy'] = sam_df_color.values[:,i] + delta_fi
    print(f'{sam_df_color.columns[i]}_noisy')

acsf814w_dust-VISTA_Y_dust_noisy
acsf814w_dust-VISTA_J_dust_noisy
acsf814w_dust-VISTA_H_dust_noisy
acsf814w_dust-VISTA_Ks_dust_noisy
acsf814w_dust-irac_ch1_dust_noisy
acsf814w_dust-irac_ch2_dust_noisy
acsf814w_dust-NIRCam_F115W_dust_noisy
acsf814w_dust-NIRCam_F150W_dust_noisy
acsf814w_dust-NIRCam_F277W_dust_noisy
acsf814w_dust-NIRCam_F444W_dust_noisy
VISTA_Y_dust-VISTA_J_dust_noisy
VISTA_Y_dust-VISTA_H_dust_noisy
VISTA_Y_dust-VISTA_Ks_dust_noisy
VISTA_Y_dust-irac_ch1_dust_noisy
VISTA_Y_dust-irac_ch2_dust_noisy
VISTA_Y_dust-NIRCam_F115W_dust_noisy
VISTA_Y_dust-NIRCam_F150W_dust_noisy
VISTA_Y_dust-NIRCam_F277W_dust_noisy
VISTA_Y_dust-NIRCam_F444W_dust_noisy
VISTA_J_dust-VISTA_H_dust_noisy
VISTA_J_dust-VISTA_Ks_dust_noisy
VISTA_J_dust-irac_ch1_dust_noisy
VISTA_J_dust-irac_ch2_dust_noisy
VISTA_J_dust-NIRCam_F115W_dust_noisy
VISTA_J_dust-NIRCam_F150W_dust_noisy
VISTA_J_dust-NIRCam_F277W_dust_noisy
VISTA_J_dust-NIRCam_F444W_dust_noisy
VISTA_H_dust-VISTA_Ks_dust_noisy
VISTA_H_dust-irac_ch1_du